# eas-3d-pattern — Feature Showcase

**eas-3d-pattern** is a Python library for working with 3D antenna radiation patterns following the [NGMN BASTA](https://www.ngmn.org/schema/basta/) JSON schema.

This notebook demonstrates the core capabilities:

| # | Feature | What it does |
|---|---------|-------------|
| 1 | Load & validate | Parse any BASTA-compliant JSON, with optional schema validation |
| 2 | Inspect metadata | Access antenna parameters (gain, frequency, tilt, beamwidth) |
| 3 | Visualize | Interactive 2D heatmap and 3D polar radiation plots |
| 4 | Beam efficiency | Calculate power distribution across configurable sectors |
| 5 | Custom sectors | Define your own rectangular regions for analysis |

---


## Setup

```bash
pip install eas-3d-pattern
```

In [ ]:
from eas_3d_pattern import (
    SAMPLE_JSON,
    AntennaPattern,
    SectorDefinition,
)


---
## 1. Load an Antenna Pattern

The library ships with sample NGMN BASTA JSON files. In practice, you would point to your own file path.

In [ ]:
# Load a sample 3D radiation pattern (non-uniform sampling)
pattern = AntennaPattern(SAMPLE_JSON[0])

# the pattern is already parsed and its ready for analysis
print(pattern)

The `print()` output shows all metadata from the JSON header: supplier, model, frequency, gain, beamwidth, tilt, and data structure info.

---
## 2. Access Antenna Parameters

All metadata is accessible as typed properties:

In [ ]:
print(f"Supplier:\t\t{pattern.supplier}")
print(f"Model:\t\t\t{pattern.antenna_model}")
print(f"Frequency:\t\t{pattern.frequency_hz / 1e6:.0f} MHz")
print(f"Gain:\t\t\t{pattern.gain_dbi:.1f} dBi")
print(f"Horizontal HPBW:\t{pattern.phi_hpbw:.1f}°")
print(f"Vertical HPBW:\t\t{pattern.theta_hpbw:.1f}°")
print(f"Front-to-Back:\t\t{pattern.front_to_back:.1f} dB")
print(f"Coordinate System:\t{pattern.coordinate_system}")
print(f"Sampling:\t\t{'Uniform' if pattern.is_uniform_sampling else 'Non-uniform'}")

---
## 3. Visualize the Radiation Pattern

### 3.1 — 2D Heatmap

The heatmap shows the normalized power pattern in dB across the sphere (θ: 0° to 180°, φ: −180° to 179°). The color scale ranges from 0 dB to −30 dB.

In [ ]:
# By default 'P_tp_dB' is plotted
pattern.plot()

You can also visualize individual components:

In [ ]:
# Co-polarized component only
pattern.plot(component_name="P_co_dB")

### 3.2 — 3D Polar Plot

For spatial intuition, the 3D view maps the radiation intensity onto a sphere. Rotate and zoom to inspect the beam shape from any angle.

In [ ]:
pattern.plot_3D()

---
## 4. Calculate Beam Efficiency

**Beam efficiency** measures how much of the total radiated power falls within a defined sector of interest (the service area) versus interference or unused regions.

The library supports sector presets based on the [NGMN BASTA V13.0](https://www.ngmn.org/wp-content/uploads/NGMN_BASTA_Recommendations-for-Base-Station-Antennas_V13.0.pdf) specification (Section 7.2.4, Table 7-1).

the sphere is partitioned into:

| Sector | Description |
|--------|-------------|
| **Service** | Main coverage area (bounded by HPBW and nominal sector width) |
| **Interference_Left** | Left of service area |
| **Interference_Right** | Right of service area |
| **Interference_Upper** | Above service area, below upper boundary |
| **Upper** | Above the interference region (toward sky) |
| **Lower** | Below 165° elevation (toward ground/mounting structure) |

Boundaries are computed dynamically from the beam peak and declared HPBW per the NGMN formulas.


In [ ]:
# Switch to NGMN BASTA V13 Type A sector preset
pattern.sector_preset = "ngmn-v13-type-a"

efficiency = pattern.calculate_beam_efficiency()

print("Beam Efficiency — NGMN BASTA V13 sectors:")
print("-" * 50)

total = sum(efficiency.values()) * 100
for sector, value in efficiency.items():
    pct = value * 100
    bar = "█" * int(pct)
    print(f"  {sector:22s} {pct:5.1f}%  {bar}")

print("-" * 50)
print(f"  {'Total':22s} {total:5.1f}%")



---
## 5. Custom Sector Definitions

You can define your own rectangular sectors to match specific deployment scenarios.

**Example:** Evaluate the left half-beam.

In [ ]:
# Create a custom sector definition without defaults
custom_sectors = SectorDefinition(load_default=False)

# Add a sector covering the left half of the main beam
custom_sectors.add_sector(
    name="Left_Half_Beam",
    theta_min=(85.0, "<="),   # from 85°
    theta_max=(165.0, "<="),  # down to 165°
    phi_min=(-60.0, "<="),    # left side
    phi_max=(0.0, "<="),      # up to boresight
)

# Add the right half for comparison
custom_sectors.add_sector(
    name="Right_Half_Beam",
    theta_min=(85.0, "<="),
    theta_max=(165.0, "<="),
    phi_min=(0.0, "<"),
    phi_max=(60.0, "<="),
)

# Calculate efficiency for our custom sectors
custom_eff = pattern.calculate_beam_efficiency(sector_definitions=custom_sectors)

print("Custom Sector Analysis:")
for sector, ratio in custom_eff.items():
    print(f"  {sector:16s} → {ratio * 100:.1f}%")

---
## 6. Compare Multiple Patterns

Load multiple patterns (e.g., different frequencies or tilts) and compare their beam efficiency side by side.

In [ ]:
import os  # for path manipulation

# Load all available sample patterns and compute NGMN beam efficiency
patterns = [AntennaPattern(f) for f in SAMPLE_JSON]

print(f"{'File':<25s} {'Freq (MHz)':>10s} {'Gain (dBi)':>10s} {'HPBW-H':>7s} {'HPBW-V':>7s} {'Service%':>9s}")
print("─" * 78)

for p in patterns:
    p.sector_preset = "ngmn-v13-type-a"
    service_pct = p.calculate_beam_efficiency()["Service"] * 100
    short_name = os.path.split(p.data_filepath)[1][:21]
    print(f"{short_name:<25s} {(p.frequency_hz/1e6):>10.0f} {p.gain_dbi:>10.1f} {p.phi_hpbw:>7.1f} {p.theta_hpbw:>7.1f} {service_pct:>9.1f}")


---
## 7. Additional Capabilities

### Directivity & Losses

In [ ]:
directivity = pattern.calculate_directivity()
losses = pattern.calculate_losses()

print(f"{"Declared Gain:":<25s} {pattern.gain_dbi:>5.2f} dBi")
print(f"{"Calculated Directivity:":<25s} {directivity:>5.2f} dBi")
print(f"{"Estimated Losses:":<25s} {losses:>5.2f} dB")

### Schema Validation

Validate any JSON file against the official NGMN BASTA schema to catch data issues early:

In [ ]:
# validate=True checks the JSON structure against the NGMN schema
validated_pattern = AntennaPattern(SAMPLE_JSON[1], validate=True)
print(f"{validated_pattern.antenna_model}, schema validation passed")

### Coordinate System Support

The library accepts all NGMN-defined coordinate systems on input and transforms them automatically to a canonical internal frame for calculations:

- **SPCS_Polar** — Standard spherical
- **SPCS_CW** / **SPCS_CCW** — Clockwise/counter-clockwise
- **SPCS_Geo** — Geographic

You don't need to pre-process your JSON files, the coordinate system transformation is handled internally with post-condition checks to catch out-of-spec data.

---

**Links:**
- PyPI: [pypi.org/project/eas-3d-pattern](https://pypi.org/project/eas-3d-pattern/)
- NGMN BASTA Schema: [ngmn.org/schema/basta](https://www.ngmn.org/schema/basta/)